In [1]:
import ewatercycle.forcing
import ewatercycle.observation.grdc
import ewatercycle.analysis
from pathlib import Path
from cartopy.io import shapereader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from rich import print
import shutil


/opt/conda/envs/ewatercycle2/lib/python3.12/site-packages/esmvalcore/experimental/_warnings.py:13: UserWarning: 
  Thank you for trying out the new ESMValCore API.
  Note that this API is experimental and may be subject to change.
  More info: https://github.com/ESMValGroup/ESMValCore/issues/498


In [ ]:
own_region = "Big_shape_file" 
if own_region == None: 
    own_region = "Big_shape_file"

path = Path.cwd()
forcing_path = path / "Forcing"    
shapeFile = forcing_path / f"{own_region}.shp"

saveLocation = forcing_path / f"{own_region}Forcing"

In [ ]:
experiment_start_time = "2001-01-01T00:00:00Z"
experiment_end_time = "2019-12-31T00:00:00Z"

In [ ]:
ERA5_forcing = ewatercycle.forcing.sources["LumpedMakkinkForcing"].generate(
    dataset="ERA5",
    start_time=experiment_start_time,
    end_time=experiment_end_time,
    shape=shapeFile.absolute(),
    
)


In [ ]:
ds = ERA5_forcing.to_xarray()

df_forcing = pd.DataFrame({
    "date": pd.to_datetime(ds["time"].values),
    "P": ds["pr"].values,
    "Ep": ds["evspsblpot"].values
})

df_forcing["P"] *= 86400
df_forcing["Ep"] *= 86400

df_forcing.head()

In [ ]:
df_q = pd.read_csv(
    "6228920_Q_Day.Cmd.txt",
    sep=";",
    comment="#",
    encoding="latin1",
    header=None,
    names=["date", "time", "Q"]
)

# remove the non-data row: YYYY-MM-DD;hh:mm; Value
df_q = df_q[df_q["date"] != "YYYY-MM-DD"].copy()

# clean columns
df_q["date"] = pd.to_datetime(df_q["date"], errors="coerce")
df_q["Q"] = pd.to_numeric(df_q["Q"], errors="coerce")

# remove missing GRDC values
df_q = df_q[(df_q["Q"] != -999.0) & (df_q["date"].notna())].copy()

# keep only what you need
df_q = df_q[["date", "Q"]]

df_q.head()

In [ ]:
df_forcing["date"] = pd.to_datetime(df_forcing["date"]).dt.normalize()
df_q["date"] = pd.to_datetime(df_q["date"]).dt.normalize()

In [ ]:
area_km2 = 4010   # temporary assumption
area_m2 = area_km2 * 1e6

print("Catchment area [km²]:", area_km2)

In [ ]:
area_km2 = 3156.19999999   # temporary assumption
area_m2 = area_km2 * 1e6

print("Catchment area [km²]:", area_km2)

In [ ]:
df_q["R"] = df_q["Q"] * 86400 / area_m2 * 1000
df_q.head()

In [ ]:
df_forcing["date"] = pd.to_datetime(df_forcing["date"]).dt.normalize()
df_q["date"] = pd.to_datetime(df_q["date"]).dt.normalize()

df = pd.merge(df_forcing, df_q[["date", "R"]], on="date", how="inner")

df_month = df.set_index("date").resample("ME").sum()
df_month.head()

In [ ]:
P_mean = df_month["P"].mean()
Ep_mean = df_month["Ep"].mean()
R_mean = df_month["R"].mean()

Ea_mean = P_mean - R_mean

print("P_mean =", P_mean)
print("Ep_mean =", Ep_mean)
print("R_mean =", R_mean)
print("Ea_mean =", Ea_mean)
print("Check closure =", P_mean - (Ea_mean + R_mean))

In [ ]:
AI = Ep_mean / P_mean
EI_obs = Ea_mean / P_mean   # same as 1 - R_mean/P_mean

print("Aridity Index (Ep/P) =", AI)
print("Evaporative Index (Ea/P) =", EI_obs)
print("Check EI =", 1 - R_mean / P_mean)

In [ ]:
phi_range = np.linspace(0.01, 5, 500)

EaP_schreiber = 1 - np.exp(-phi_range)
EaP_oldekop = phi_range * np.tanh(1 / phi_range)
EaP_budyko = np.sqrt(phi_range * np.tanh(1/phi_range) * (1 - np.exp(-phi_range)))
EaP_turc = 1 / np.sqrt(0.9 + (1/phi_range)**2)

plt.figure(figsize=(8, 6))
plt.hlines(1, xmin=0, xmax=5, colors='black', linestyles='--', label='Water limit')
plt.plot(phi_range, phi_range, 'k--', label='Energy limit')
plt.plot(phi_range, EaP_budyko, label='Budyko (1948)')
plt.scatter(AI, EI_obs, color='red', s=80, label='Ter basin')

plt.xlabel("Aridity Index (Ep/P)")
plt.ylabel("Evaporative Index (Ea/P)")
plt.title("Budyko Framework for the Ter basin")
plt.xlim(0, 5)
plt.ylim(0, 1.2)
plt.grid(True)
plt.legend()
plt.show()